In [1]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain

/Users/root1/Documents/maestria/ai_uaq/nlp/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password="test1234",
)

In [3]:
graph.query("RETURN 1 AS ok")

[{'ok': 1}]

In [4]:
import json
import pandas as pd 
import numpy as np
pd.options.display.max_columns = None


In [5]:
import getpass
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [7]:
from langchain_core.prompts import PromptTemplate


In [8]:
GENERIC_CYPHER_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template="""
Eres experto en Cypher para Neo4j 5.
Devuelve **solo** un bloque:
```cypher
<CONSULTA CYPHER>
```
Reglas:
	•	Para “caminando / a pie / cerca / distancia”, usar arista no dirigida :IS_WALKING. 
    • Si el usuario pide los restaurantes que están caminando de un restaurante, usa la siguietne query como base: 
     	 MATCH (v:Restaurant) WHERE toLower(v.title)=toLower($TITLE)
		MATCH (v)-[e:IS_WALKING]-(n:Restaurant)
		RETURN n.title AS name, round(e.meters) AS meters, round(e.time_min,1) AS minutes, n.address AS address
		ORDER BY meters ASC
	•	Ancla por título de restaurante:
		MATCH (v:Restaurant) WHERE toLower(v.title) = toLower($TITLE)
	•	Si mencionan “Querétaro/Qro/Queretarock”, hace referencia a "Santiago de Querétaro".
	•	Ordena y limita resultados razonablemente
Esquema:
{schema}

Pregunta:
{question}
"""
)


In [9]:
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    return_intermediate_steps=True,
    top_k=20,
    allow_dangerous_requests=True,
    cypher_prompt=GENERIC_CYPHER_PROMPT,

)

In [10]:
# -*- coding: utf-8 -*-
import folium
from folium.plugins import MarkerCluster

def _pick(d, keys):
    """Devuelve la primera key presente y no None."""
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return None

def _norm_row_to_restaurant(row):
    """
    Normaliza una fila del contexto a un dict:
    {title, lat, lon, address, rating}
    Soporta:
      - row['r'] / row['n'] / row['r1'] / row['r2'] como mapas de propiedades
      - columnas planas 'r.title', 'r.latitude', etc.
    """
    # 1) Caso: trae un nodo como dict con props
    for node_key in ("r", "n", "r1", "r2", "v"):
        node = row.get(node_key)
        if isinstance(node, dict):
            return {
                "title": node.get("title") or node.get("name"),
                "lat": node.get("latitude"),
                "lon": node.get("longitude"),
                "address": node.get("address"),
                "rating": node.get("rating"),
            }

    # 2) Caso: columnas sueltas / aliasados
    title = _pick(row, ["title", "name", "r.title", "n.title", "r2.title"])
    lat   = _pick(row, ["latitude", "r.latitude", "n.latitude", "r2.latitude"])
    lon   = _pick(row, ["longitude","r.longitude","n.longitude","r2.longitude"])
    addr  = _pick(row, ["address","r.address","n.address","r2.address"])
    rate  = _pick(row, ["rating","r.rating","n.rating","r2.rating"])
    return {"title": title, "lat": lat, "lon": lon, "address": addr, "rating": rate}

def _fetch_by_title(graph, title):
    """Consulta de respaldo: obtiene lat/lon por título si faltan en el contexto."""
    cy = """
    MATCH (r:Restaurant)
    WHERE toLower(r.title) = toLower($title)
    RETURN r.title AS title, r.latitude AS lat, r.longitude AS lon,
           r.address AS address, r.rating AS rating
    """
    out = []
    for row in graph.query(cy, {"title": title}):
        out.append({
            "title": row.get("title"),
            "lat": row.get("lat"),
            "lon": row.get("lon"),
            "address": row.get("address"),
            "rating": row.get("rating"),
        })
    return out

def _plot_restaurants_folium(items, outfile="restaurants_map.html", zoom_start=13):
    """Crea un mapa Folium con MarkerCluster y lo guarda en HTML."""
    coords = [(it["lat"], it["lon"]) for it in items
              if isinstance(it.get("lat"), (int, float)) and isinstance(it.get("lon"), (int, float))]
    if not coords:
        return None

    # Centro: promedio simple
    lat_c = sum(lat for lat, _ in coords) / len(coords)
    lon_c = sum(lon for _, lon in coords) / len(coords)

    m = folium.Map(location=[lat_c, lon_c], zoom_start=zoom_start, control_scale=True)
    cluster = MarkerCluster().add_to(m)

    for it in items:
        lat, lon = it.get("lat"), it.get("lon")
        if not isinstance(lat, (int, float)) or not isinstance(lon, (int, float)):
            continue
        title = it.get("title") or "Restaurante"
        addr  = it.get("address") or ""
        rate  = it.get("rating")
        popup = f"<b>{title}</b>"
        if addr: popup += f"<br>{addr}"
        if rate is not None: popup += f"<br>⭐ {rate}"
        folium.Marker(
            location=[lat, lon],
            tooltip=title,
            popup=folium.Popup(popup, max_width=320),
        ).add_to(cluster)

    m.save(outfile)
    return outfile, m

def query_and_plot_restaurants(chain, graph, payload, outfile="restaurants_map.html", verbose=True):
    """
    Ejecuta el chain, extrae filas del contexto y genera un mapa Folium si hay restaurantes.
    - chain: GraphCypherQAChain (return_intermediate_steps=True)
    - graph: Neo4jGraph (para la consulta de respaldo por título)
    - payload: dict para chain.invoke({"query": ...})
    - outfile: ruta del HTML a guardar
    """
    res = chain.invoke(payload)

    # 1) Captura filas crudas del contexto
    steps = res.get("intermediate_steps") or []
    context_rows = steps[-1].get("context") if steps else []
    if verbose:
        print(f"[INFO] Filas en contexto: {len(context_rows)}")

    # 2) Normaliza a items {title,lat,lon,address,rating}
    items = []
    titles_missing_coords = []
    for row in (context_rows or []):
        item = _norm_row_to_restaurant(row)
        if item.get("lat") is None or item.get("lon") is None:
            # Si solo tengo el título, intenta fetch por título
            if item.get("title"):
                titles_missing_coords.append(item["title"])
        else:
            items.append(item)

    # 3) Backfill por título si faltan coords
    for t in titles_missing_coords:
        items.extend(_fetch_by_title(graph, t))

    # 4) Pinta si hay coordenadas
    plotted = _plot_restaurants_folium(items, outfile=outfile)
    if plotted:
        path, _map = plotted
        if verbose:
            print(f"[OK] Mapa guardado en: {path}")
    else:
        if verbose:
            print("[WARN] No hay restaurantes con coordenadas para mapear.")

    return res  # regresa la respuesta original del chain por si la quieres imprimir

In [11]:
# 2) Consulta general (ej., top por rating en Querétaro) y plot igual
res = query_and_plot_restaurants(
    chain,
    graph,
    {"query": "Dame 10 restaurantes con mejor rating en Querétaro"},
    outfile="top_rating_qro.html"
)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (r:Restaurant)-[:LOCATED_IN]->(c:City)
WHERE toLower(c.name) = "santiago de querétaro"
RETURN r.title AS name, r.rating AS rating
ORDER BY rating DESC
LIMIT 10

Full Context:
[{'name': "K'antal vegano", 'rating': 5.0}, {'name': 'La Planta Punk', 'rating': 5.0}, {'name': 'Mister Rosso Kitchenette', 'rating': 4.9}, {'name': 'COCINA LA CAYENA', 'rating': 4.9}, {'name': 'Antojos Veganos by VegCo', 'rating': 4.8}, {'name': 'Tacogreen', 'rating': 4.8}, {'name': 'Yurei Roll', 'rating': 4.8}, {'name': 'Clorofila vegetarian restaurant', 'rating': 4.8}, {'name': 'Al Sur Cocina Diversa', 'rating': 4.8}, {'name': 'Carbónico', 'rating': 4.7}]

> Finished chain.
[INFO] Filas en contexto: 10
[OK] Mapa guardado en: top_rating_qro.html
